# Task
Predict the Nifty 50 price direction using BERT embeddings derived from the OHLCV data in "NIFTY50.csv", including data loading and preprocessing, textual description generation, BERT embedding extraction, classification model training and evaluation, and result summarization.

## Load and Preprocess Data

### Subtask:
Load the `NIFTY50.csv` file into a pandas DataFrame. Parse the 'Date' column and set it as the index. Handle any missing values if present and select the 'Open', 'High', 'Low', 'Close', and 'Adj Close' columns.


In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv('/content/NIFTY50.csv')

# Convert 'Date' to datetime and set as index
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)

# Check for missing values
print("Missing values before handling:")
print(df.isnull().sum())

# Handle missing values (e.g., forward fill for time series data)
df.ffill(inplace=True)
print("\nMissing values after handling:")
print(df.isnull().sum())

# Select the required columns
df = df[['Open', 'High', 'Low', 'Close', 'Adj Close']]

# Display the first few rows and info of the processed DataFrame
print("\nDataFrame head after processing:")
print(df.head())
print("\nDataFrame info after processing:")
df.info()

Missing values before handling:
Open         31
High         31
Low          31
Close        31
Adj Close    31
Volume       31
dtype: int64

Missing values after handling:
Open         0
High         0
Low          0
Close        0
Adj Close    0
Volume       0
dtype: int64

DataFrame head after processing:
                   Open         High          Low        Close    Adj Close
Date                                                                       
2007-09-17  4518.450195  4549.049805  4482.850098  4494.649902  4494.649902
2007-09-18  4494.100098  4551.799805  4481.549805  4546.200195  4546.200195
2007-09-19  4550.250000  4739.000000  4550.250000  4732.350098  4732.350098
2007-09-20  4734.850098  4760.850098  4721.149902  4747.549805  4747.549805
2007-09-21  4752.950195  4855.700195  4733.700195  4837.549805  4837.549805

DataFrame info after processing:
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 4153 entries, 2007-09-17 to 2024-07-15
Data columns (total 5 columns):


## Define Target Variable (Price Direction)

### Subtask:
Create the target variable for prediction. Define the price direction for the next day: 1 if the next day's 'Close' price is higher than the current day's 'Close' price, and 0 otherwise.


In [ ]:
import numpy as np

# Calculate the 'next_day_close' price
df['next_day_close'] = df['Close'].shift(-1)

# Create the 'price_direction' target variable
df['price_direction'] = np.where(df['next_day_close'] > df['Close'], 1, 0)

# Drop rows with NaN values (the last row due to shifting)
df.dropna(inplace=True)

# Display the last few rows to show the new columns
print("\nDataFrame tail after adding price_direction:")
print(df.tail())

# Display the count of each class in price_direction
print("\nValue counts for 'price_direction':")
print(df['price_direction'].value_counts())


DataFrame tail after adding price_direction:
                    Open          High           Low         Close  \
Date                                                                 
2024-07-08  24329.449219  24344.599609  24240.550781  24320.550781   
2024-07-09  24351.000000  24443.599609  24331.900391  24433.199219   
2024-07-10  24459.849609  24461.050781  24141.800781  24324.449219   
2024-07-11  24396.550781  24402.650391  24193.750000  24315.949219   
2024-07-12  24387.949219  24592.199219  24331.150391  24502.150391   

               Adj Close  next_day_close  price_direction  
Date                                                       
2024-07-08  24320.550781    24433.199219                1  
2024-07-09  24433.199219    24324.449219                0  
2024-07-10  24324.449219    24315.949219                0  
2024-07-11  24315.949219    24502.150391                1  
2024-07-12  24502.150391    24502.150391                0  

Value counts for 'price_direction':
price_

## Generate Textual Descriptions for BERT

### Subtask:
For each trading day, convert the numerical OHLC and Adjusted Close data into a descriptive sentence or paragraph. This text will serve as the input for the `bert-base-uncased` model. For example, a sentence might describe the daily open, high, low, close, and adjusted close values, along with daily percentage changes.


In [ ]:
import numpy as np

# Calculate daily percentage changes for OHLC and Adj Close
df['Open_pct_change'] = df['Open'].pct_change() * 100
df['High_pct_change'] = df['High'].pct_change() * 100
df['Low_pct_change'] = df['Low'].pct_change() * 100
df['Close_pct_change'] = df['Close'].pct_change() * 100
df['Adj_Close_pct_change'] = df['Adj Close'].pct_change() * 100

# Fill NaN values from pct_change (first row) with 0
df.fillna(0, inplace=True)

# Function to create a descriptive sentence for each day
def create_description(row):
    return (
        f"""On this trading day, the Nifty 50 opened at {row['Open']:.2f} "
        f"(changed {row['Open_pct_change']:.2f}%), reached a high of {row['High']:.2f} "
        f"(changed {row['High_pct_change']:.2f}%), a low of {row['Low']:.2f} "
        f"(changed {row['Low_pct_change']:.2f}%), and closed at {row['Close']:.2f} "
        f"(changed {row['Close_pct_change']:.2f}%). The adjusted close was {row['Adj Close']:.2f} "
        f"(changed {row['Adj_Close_pct_change']:.2f}%)."""
    )

# Apply the function to create the 'text_description' column
df['text_description'] = df.apply(create_description, axis=1)

# Display the first few rows with the new 'text_description' column
print("\nDataFrame head with text_description:")
print(df[['Open', 'Close', 'Open_pct_change', 'Close_pct_change', 'text_description']].head())


DataFrame head with text_description:
                   Open        Close  Open_pct_change  Close_pct_change  \
Date                                                                      
2007-09-17  4518.450195  4494.649902         0.000000          0.000000   
2007-09-18  4494.100098  4546.200195        -0.538904          1.146926   
2007-09-19  4550.250000  4732.350098         1.249414          4.094626   
2007-09-20  4734.850098  4747.549805         4.056922          0.321187   
2007-09-21  4752.950195  4837.549805         0.382274          1.895715   

                                             text_description  
Date                                                           
2007-09-17  On this trading day, the Nifty 50 opened at 45...  
2007-09-18  On this trading day, the Nifty 50 opened at 44...  
2007-09-19  On this trading day, the Nifty 50 opened at 45...  
2007-09-20  On this trading day, the Nifty 50 opened at 47...  
2007-09-21  On this trading day, the Nifty 50 opene

## Extract BERT Embeddings

### Subtask:
Install the `transformers` library if not already installed. Load the `bert-base-uncased` tokenizer and model. Tokenize the generated textual descriptions and then feed them through the BERT model to obtain contextualized embeddings. We will extract the `[CLS]` token's embedding for each sentence as the fixed-size representation of that day's financial data.


In [ ]:
import sys
!{sys.executable} -m pip install transformers torch
print("Transformers and Torch libraries installed.")

Transformers and Torch libraries installed.


**Reasoning**:
Now that `transformers` and `torch` are installed, I will import the necessary components, load the `bert-base-uncased` tokenizer and model, tokenize the `text_description` column in batches, and extract the `[CLS]` token embeddings. These embeddings will then be stored in a list and converted to a NumPy array for further use.



In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel

# --- Start of re-included data loading and preprocessing steps ---
# Load the dataset
df = pd.read_csv('/content/NIFTY50.csv')

# Convert 'Date' to datetime and set as index
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)

# Handle missing values (e.g., forward fill for time series data)
df.ffill(inplace=True)

# Select the required columns
df = df[['Open', 'High', 'Low', 'Close', 'Adj Close']]

# Calculate the 'next_day_close' price
df['next_day_close'] = df['Close'].shift(-1)

# Create the 'price_direction' target variable
df['price_direction'] = np.where(df['next_day_close'] > df['Close'], 1, 0)

# Drop rows with NaN values (the last row due to shifting)
df.dropna(inplace=True)

# Calculate daily percentage changes for OHLC and Adj Close
df['Open_pct_change'] = df['Open'].pct_change() * 100
df['High_pct_change'] = df['High'].pct_change() * 100
df['Low_pct_change'] = df['Low'].pct_change() * 100
df['Close_pct_change'] = df['Close'].pct_change() * 100
df['Adj_Close_pct_change'] = df['Adj Close'].pct_change() * 100

# Fill NaN values from pct_change (first row) with 0
df.fillna(0, inplace=True)

# Function to create a descriptive sentence for each day
def create_description(row):
    return (
        f"""On this trading day, the Nifty 50 opened at {row['Open']:.2f} "
        f"(changed {row['Open_pct_change']:.2f}%), reached a high of {row['High']:.2f} "
        f"(changed {row['High_pct_change']:.2f}%), a low of {row['Low']:.2f} "
        f"(changed {row['Low_pct_change']:.2f}%), and closed at {row['Close']:.2f} "
        f"(changed {row['Close_pct_change']:.2f}%). The adjusted close was {row['Adj Close']:.2f} "
        f"(changed {row['Adj_Close_pct_change']:.2f}%)."""
    )

# Apply the function to create the 'text_description' column
df['text_description'] = df.apply(create_description, axis=1)
# --- End of re-included data loading and preprocessing steps ---

# Load pre-trained BERT tokenizer and model
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
model = AutoModel.from_pretrained('bert-base-uncased')

# Define batch size
batch_size = 32

# List to store BERT embeddings
bert_embeddings = []

# Iterate through text descriptions in batches
for i in range(0, len(df), batch_size):
    batch_descriptions = df['text_description'].iloc[i:i + batch_size].tolist()

    # Tokenize the batch
    encoded_input = tokenizer(batch_descriptions, padding=True, truncation=True, return_tensors='pt', max_length=512)

    # Move model to CPU (if not already there) and ensure input tensors are on the same device
    # Using CPU as no GPU is explicitly requested or checked for. If GPU was available, it would be 'cuda'
    model.cpu()
    input_ids = encoded_input['input_ids'].cpu()
    attention_mask = encoded_input['attention_mask'].cpu()

    # Generate embeddings
    with torch.no_grad():
        output = model(input_ids, attention_mask=attention_mask)

    # Extract [CLS] token embeddings (first token in the last hidden state)
    # The [CLS] token is usually at index 0 of the sequence output
    batch_cls_embeddings = output.last_hidden_state[:, 0, :].numpy()
    bert_embeddings.append(batch_cls_embeddings)

# Concatenate all batch embeddings into a single NumPy array
bert_embeddings_array = np.concatenate(bert_embeddings, axis=0)

print(f"Shape of BERT embeddings: {bert_embeddings_array.shape}")
print("BERT embeddings successfully extracted.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Shape of BERT embeddings: (4152, 768)
BERT embeddings successfully extracted.


## Prepare Data for Classification

### Subtask:
Align the extracted BERT embeddings with the corresponding target price direction labels. Remove any rows with NaN values that might result from shifting the target variable. Split the dataset into training and testing sets to prepare for model training.


In [ ]:
from sklearn.model_selection import train_test_split

# Create DataFrame X from BERT embeddings
X = pd.DataFrame(bert_embeddings_array)

# Extract the target variable y
y = df['price_direction']

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

print("Data split into training and testing sets successfully.")

Shape of X_train: (3321, 768)
Shape of X_test: (831, 768)
Shape of y_train: (3321,)
Shape of y_test: (831,)
Data split into training and testing sets successfully.


## Train and Evaluate Classifier

### Subtask:
Train a suitable classification model (e.g., Logistic Regression, RandomForest Classifier, or a simple Multi-Layer Perceptron) using the BERT embeddings as features and the defined price direction as the target variable. Evaluate the model's performance on the test set using metrics such as accuracy, precision, recall, and F1-score.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Instantiate Logistic Regression model
# Using 'liblinear' solver for smaller datasets and 'lbfgs' for larger ones.
# max_iter is increased for convergence.
model = LogisticRegression(max_iter=1000, solver='liblinear', random_state=42)

# Train the model
print("Training Logistic Regression model...")
model.fit(X_train, y_train)
print("Model training complete.")

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"\nModel Evaluation on Test Set:")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

Training Logistic Regression model...
Model training complete.

Model Evaluation on Test Set:
Accuracy: 0.5114
Precision: 0.5179
Recall: 0.6471
F1-Score: 0.5753
